# Llama-2 French-to-English Translation Pipeline

This notebook implements an end-to-end pipeline for fine-tuning a Large Language Model (TinyLlama/Llama-2) for machine translation. The pipeline follows these key phases:
1.  **Environment Setup**: Installing dependencies and verifying hardware.
2.  **Configuration**: Centralizing model names and hyperparameters.
3.  **Data Acquisition**: Loading the translation dataset.
4.  **Preprocessing**: Tokenization and data formatting.
5.  **Model Preparation**: Quantization (4-bit) and LoRA configuration.
6.  **Training**: Fine-tuning using SFTTrainer.
7.  **Inference**: Testing the model on French-to-English translation tasks.

## Phase 1: Environment Setup

First, we install the necessary libraries for fine-tuning: `transformers`, `trl`, `peft`, `accelerate`, and `bitsandbytes`.

In [ ]:
!pip install -q transformers trl datasets accelerate peft bitsandbytes

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

### Library Imports

In [ ]:
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import SFTTrainer
import os
import torch

## Phase 2: Pipeline Configuration

We centralize model names and training hyperparameters to make the pipeline easily configurable.

In [ ]:
# Model configuration
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_name = "kaitchup/opus-French-to-English"
new_model_name = "tinyllama-fr-en-translator"

# Training hyperparameters
output_dir = "./results"
num_train_epochs = 1
max_steps = 3000
per_device_train_batch_size = 16
gradient_accumulation_steps = 1
learning_rate = 1e-4
max_seq_length = 120

## Phase 3: Data Acquisition

We load the French-to-English translation dataset from Hugging Face.

In [ ]:
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

# Sample display
print("Sample data point:")
print(dataset['train'][0])

## Phase 4: Data Preprocessing

We initialize the tokenizer and prepare the data formatting.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True, add_eos_token=True)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "left"
print("Tokenizer initialized.")

## Phase 5: Model Preparation

We configure 4-bit quantization using `bitsandbytes` and set up LoRA (Low-Rank Adaptation) for efficient fine-tuning.

In [ ]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name, 
    quantization_config=bnb_config, 
    device_map={"": 0}
)

model = prepare_model_for_kbit_training(model)

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["down_proj", "up_proj", "gate_proj"]
)

## Phase 6: Training (Fine-Tuning)

We define the training arguments and initialize the `SFTTrainer` for supervised fine-tuning.

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="steps",
    optim="paged_adamw_8bit",
    save_steps=500,
    logging_steps=100,
    learning_rate=learning_rate,
    eval_steps=500,
    fp16=True, 
    per_device_train_batch_size=per_device_train_batch_size,
    warmup_steps=100,
    max_steps=max_steps,
    lr_scheduler_type="linear",
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments
)

In [ ]:
# Memory stats before training
print(torch.cuda.memory_summary(device=None, abbreviated=True))

# trainer.train()  # Uncomment to start training

## Phase 7: Inference & Translation Test

After training, we test the model by providing a French sentence and generating its English translation.

In [ ]:
def translate(text, model, tokenizer):
    prompt = f"{text} ###>"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        num_beams=5, 
        early_stopping=True
    )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("###>")[1].strip() if "###>" in result else result

# Test sentence
test_sentence = "Tu es le seul client du magasin."
print(f"French: {test_sentence}")
# print(f"English: {translate(test_sentence, model, tokenizer)}") # Uncomment after training